In [ ]:
# !pip install git+https://github.com/monash-emu/summer3wip.git

In [ ]:
import numpy as np
from plotly import graph_objects as go

# Import pandas for data wrangling and use plotly module for interactive visualisation
import pandas as pd
pd.options.plotting.backend = "plotly"

# Import various components of our "summer" platform for building models of epidemics
from summer3.graph import defer, Parameter, CompartmentValues
from summer3.epi import Stratification, CompartmentMap, \
    ManagedArray, CategoryData, TransitionFlow, CompartmentalEpiModel, \
    build_istate, dti_to_epoch

In [ ]:
population = 1e6  
seed = 1.0
start_time = 0
end_time = 50
model_comps = ["susceptible", "infectious", "recovered"]
infect_comps = ["infectious"]
disease_state = Stratification("disease_state", model_comps)
humans = CompartmentMap.new(disease_state)
epi_model = CompartmentalEpiModel(humans, range(start_time, end_time))
start_pop = [population - seed, seed, 0.0]
epi_model.set_initial_population(CategoryData(disease_state.categories(), np.array(start_pop)))

def iprocess(compartment_values: ManagedArray, contact_rate: float, infectious_compartments: tuple):
    n_infectious = compartment_values.query(infectious_compartments).sum()
    n_population = compartment_values.sum()
    infectious_prevalence = n_infectious / n_population
    return contact_rate * infectious_prevalence

recovery = TransitionFlow("recovery", disease_state["infectious"], disease_state["recovered"], Parameter("recovery_rate", 0.0))
epi_model.add_flow(recovery)
base_parameters = {"effective_contact_rate": 1.2, "recovery_rate": 0.2}

In [ ]:
contact_rate = Parameter("effective_contact_rate", 0.0) * (1.0 - Parameter("lockdown_effect", 0.0))
foi = defer(iprocess)(CompartmentValues, contact_rate, disease_state["infectious"])
infection = TransitionFlow("infection", disease_state["susceptible"], disease_state["infectious"], foi)
epi_model.add_flow(infection)

In [ ]:
# Effects vary from zero to one in 10% increments
lockdown_effects = np.arange(0.0, 1.0, 0.1)

# Get ready to get the outputs
outputs = pd.DataFrame(columns=lockdown_effects)

# Cycle through the effects
for effect in lockdown_effects:

    # Add the lockdown effect to the base parameter set and run
    results = epi_model.run(base_parameters | {"lockdown_effect": effect})

    # Collate the results
    outputs[effect] = results["compartments"].to_pandas_df()["infectious"]

In [ ]:
fig = go.Figure()
legend_format = {"title": "lockdown effect"}
xaxis_format = {"title": "days"}
for c in outputs.columns:
    colour = f"rgba({c}, 0, {1.0 - c}, 1)"
    fig.add_trace(go.Scatter(x=outputs.index, y=outputs[c], mode="lines", name=round(c, 1), line={"color": colour}))
fig.update_layout(title="effect of lockdown severity on the epidemic", legend=legend_format, xaxis=xaxis_format, yaxis={"title": "infection prevalence"})

In [ ]:
# Calculate the cumulative numbers from the daily
cum_outputs = outputs.cumsum()

# Plot
cum_fig = go.Figure()
for c in cum_outputs.columns:
    colour = f"rgba({c}, 0, {1.0 - c}, 1)"
    cum_fig.add_trace(go.Scatter(x=cum_outputs.index, y=cum_outputs[c], mode="lines", name=round(c, 1), line={"color": colour}))
cum_fig.update_layout(title="effect of lockdown severity on cumulative cases", legend=legend_format, xaxis=xaxis_format, yaxis={"title": "cumulative cases"})

In [ ]:
# Get the cumulative cases by the last day for each lockdown severity value
cum_cases_day50 = cum_outputs.iloc[-1]

# Plot
nonlinear_fig = go.Figure()
nonlinear_fig.add_trace(go.Scatter(x=cum_outputs.columns, y=cum_cases_day50))
nonlinear_fig.update_layout(title="cumulative cases at 50 days by lockdown effect", xaxis={"title": "lockdown effect"}, yaxis={"title": "cumulative cases"})